# Regular RAG Implementation

This notebook demonstrates a standard Retrieval-Augmented Generation (RAG) pipeline using LangChain. 
It covers:
1.  Setting up the environment and API keys.
2.  Loading a PDF document.
3.  Splitting the document into chunks.
4.  Creating embeddings and storing them in a vector database (Chroma).
5.  Setting up a retriever.
6.  Constructing a RAG chain with a Groq-hosted LLM.
7.  Querying the system.

In [ ]:
# Installation
# Install necessary packages. 
# python-dotenv is added for local environment variable management.
%pip install --pre -U langchain langchain-openai langchain-Groq langchain-community pypdf chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/

In [ ]:
# Setup Environment Variables
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# Ensure LANGCHAIN_API_KEY is set in your .env file
os.environ["LANGCHAIN_PROJECT"] = "regular-rag"

if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY not found in environment.")

## 1. Import Libraries
Import the necessary LangChain components for document loading, vector storage, embeddings, and chat models.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## 2. Load Document
Load the PDF document that will serve as the knowledge base. 
*Ensure the file exists at the specified path.*

In [ ]:
# Loading document
# Replace 'Aganti.pdf' with the path to your PDF file.
pdf_path = "Aganti.pdf" 

if os.path.exists(pdf_path):
    pdf_loader = PyPDFLoader(pdf_path)
    docs = pdf_loader.load()
    print(f"Loaded {len(docs)} pages.")
    # Show first page metadata
    print(docs[0].metadata)
else:
    print(f"File not found: {pdf_path}. Please upload the file or update the path.")
    docs = []

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Beyond Static Retrieval: The Rise of Agentic RAG', 'source': '/content/ِAganti.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='Beyond  Static  Retrieval:  The  Rise  of  \nAgentic\n \nRAG\n \nWhitepaper  |  Technical  Architecture  Series  \nAuthor:  Lead  AI  Research  Scientist  &  Technical  Architect  \nDate:  October  2023  (Revised  for  Current  State  of  the  Art)  \nAbstract  \nThe  landscape  of  Generative  AI  is  undergoing  a  fundamental  phase  shift.  While  the  initial  wave  \nof\n \nLarge\n \nLanguage\n \nModels\n \n(LLMs)\n \ndemonstrated\n \nthe\n \npower\n \nof\n \nprobabilistic\n \ntoken\n \ngeneration,\n \nand\n \nthe\n \nsubsequent\n \nadoption\n \nof\n \nRetrieval-Augmented\n \nGeneration\n \n(RAG)\n \nsolved\n \nkey\n \nissues\n \nregarding\n \nknowledge\n \ngrounding,\n \nboth\n \narchitectures\n \nremain\n \nfundame

In [5]:
docs[0].metadata

{'producer': 'Skia/PDF m146 Google Docs Renderer',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'Beyond Static Retrieval: The Rise of Agentic RAG',
 'source': '/content/ِAganti.pdf',
 'total_pages': 7,
 'page': 0,
 'page_label': '1'}

In [6]:
docs[0].page_content

'Beyond  Static  Retrieval:  The  Rise  of  \nAgentic\n \nRAG\n \nWhitepaper  |  Technical  Architecture  Series  \nAuthor:  Lead  AI  Research  Scientist  &  Technical  Architect  \nDate:  October  2023  (Revised  for  Current  State  of  the  Art)  \nAbstract  \nThe  landscape  of  Generative  AI  is  undergoing  a  fundamental  phase  shift.  While  the  initial  wave  \nof\n \nLarge\n \nLanguage\n \nModels\n \n(LLMs)\n \ndemonstrated\n \nthe\n \npower\n \nof\n \nprobabilistic\n \ntoken\n \ngeneration,\n \nand\n \nthe\n \nsubsequent\n \nadoption\n \nof\n \nRetrieval-Augmented\n \nGeneration\n \n(RAG)\n \nsolved\n \nkey\n \nissues\n \nregarding\n \nknowledge\n \ngrounding,\n \nboth\n \narchitectures\n \nremain\n \nfundamentally\n \npassive.\n \nStandard\n \nRAG\n \npipelines\n \nfunction\n \nas\n \nsophisticated\n \nsearch\n \nengines\n \nattached\n \nto\n \na\n \ngenerator—linear,\n \nbrittle,\n \nand\n \nincapable\n \nof\n \nintrospection.\n \nThis  whitepaper  explores  the  emerg

## 3. Split Text
Split the loaded documents into smaller chunks to fit within the context window of the LLM and to improve retrieval accuracy.

In [7]:
# Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200,
)
splits = text_splitter.split_documents(docs)
splits

[Document(metadata={'producer': 'Skia/PDF m146 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Beyond Static Retrieval: The Rise of Agentic RAG', 'source': '/content/ِAganti.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='Beyond  Static  Retrieval:  The  Rise  of  \nAgentic\n \nRAG\n \nWhitepaper  |  Technical  Architecture  Series  \nAuthor:  Lead  AI  Research  Scientist  &  Technical  Architect  \nDate:  October  2023  (Revised  for  Current  State  of  the  Art)  \nAbstract  \nThe  landscape  of  Generative  AI  is  undergoing  a  fundamental  phase  shift.  While  the  initial  wave  \nof\n \nLarge\n \nLanguage\n \nModels\n \n(LLMs)\n \ndemonstrated\n \nthe\n \npower\n \nof\n \nprobabilistic\n \ntoken\n \ngeneration,\n \nand\n \nthe\n \nsubsequent\n \nadoption\n \nof\n \nRetrieval-Augmented\n \nGeneration\n \n(RAG)\n \nsolved\n \nkey\n \nissues\n \nregarding\n \nknowledge\n \ngrounding,\n \nboth\n \narchitectures\n \nremain\n \nfundame

In [8]:
splits[0].metadata

{'producer': 'Skia/PDF m146 Google Docs Renderer',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'Beyond Static Retrieval: The Rise of Agentic RAG',
 'source': '/content/ِAganti.pdf',
 'total_pages': 7,
 'page': 0,
 'page_label': '1'}

In [9]:
splits[0].page_content

'Beyond  Static  Retrieval:  The  Rise  of  \nAgentic\n \nRAG\n \nWhitepaper  |  Technical  Architecture  Series  \nAuthor:  Lead  AI  Research  Scientist  &  Technical  Architect  \nDate:  October  2023  (Revised  for  Current  State  of  the  Art)  \nAbstract  \nThe  landscape  of  Generative  AI  is  undergoing  a  fundamental  phase  shift.  While  the  initial  wave  \nof\n \nLarge\n \nLanguage\n \nModels\n \n(LLMs)\n \ndemonstrated\n \nthe\n \npower\n \nof\n \nprobabilistic\n \ntoken\n \ngeneration,\n \nand\n \nthe\n \nsubsequent\n \nadoption\n \nof\n \nRetrieval-Augmented\n \nGeneration\n \n(RAG)\n \nsolved\n \nkey\n \nissues\n \nregarding\n \nknowledge\n \ngrounding,\n \nboth\n \narchitectures\n \nremain\n \nfundamentally\n \npassive.\n \nStandard\n \nRAG\n \npipelines\n \nfunction\n \nas\n \nsophisticated\n \nsearch\n \nengines\n \nattached\n \nto\n \na\n \ngenerator—linear,\n \nbrittle,\n \nand\n \nincapable\n \nof\n \nintrospection.\n \nThis  whitepaper  explores  the  emerg

## 4. Embeddings & Vector Store
Initialize the embedding model (OpenAI in this case) and create a Chroma vector store from the document chunks. 
This step converts text into vector representations for semantic search.

In [ ]:
# Embeddings
# Using OpenAI Embeddings
if OPENAI_API_KEY:
    embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
    
    # Vector store
    # Creating a transient ChromaDB instance (in-memory)
    vector_database = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name="basic-rag",
    )

    # Retriever
    # Using k=6 to retrieve the top 6 most relevant chunks
    retriever = vector_database.as_retriever(
        search_kwargs={"k": 6},
    )
else:
    print("OPENAI_API_KEY is not set. Cannot initialize embeddings.")

## 5. Retrieval Test
Test the retriever to ensure it returns relevant documents for a sample query.

In [11]:
retriever.invoke("what is Rag")

[Document(metadata={'creationdate': '', 'creator': 'PyPDF', 'total_pages': 7, 'page': 1, 'source': '/content/ِAganti.pdf', 'page_label': '2', 'title': 'Beyond Static Retrieval: The Rise of Agentic RAG', 'producer': 'Skia/PDF m146 Google Docs Renderer'}, page_content='Generation)\n \nRetrieval-Augmented  Generation  (RAG)  emerged  as  the  standard  architectural  solution  to  the  \nparametric\n \nmemory\n \nproblem.\n \nBy\n \ndecoupling\n \nknowledge\n \nfrom\n \nreasoning,\n \nRAG\n \nallows\n \nthe\n \nLLM\n \nto\n \nact\n \nas\n \na\n \nprocessor\n \nfor\n \nexternal\n \ndata.\n \n2.1  Technical  Anatomy  of  Standard  RAG  \nThe  standard  RAG  architecture  introduces  a  non-parametric  memory  component—typically  a  \nVector\n \nDatabase—into\n \nthe\n \ninference\n \npipeline.\n ●  Vector  Databases:  Infrastructure  such  as  Pinecone,  Milvus,  or  Weaviate  stores  \nhigh-dimensional\n \nvector\n \nrepresentations\n \nof\n \ntext.\n \nThese\n \ndatabases\n \nutilize\n \

## 6. RAG Chain
Construct the RAG pipeline.
1.  **Prompt Template**: Define how inputs and context are formatted for the LLM.
2.  **LLM**: Initialize the ChatGroq model (using a powerful open-source model like `openai/gpt-oss-120b` or Llama 3).
3.  **Chain**: Connect the retriever, formatter, prompt, model, and output parser.

In [12]:
prompt_template= ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            """
            You are a helpful AI assistant. Answer the user's question based ONLY on the provided context below.

            Guidelines:
            1. If the answer is not in the context, say "I don't know".
            2. **Citation Requirement:** At the end of your answer, you MUST cite the source document names or page numbers you used (e.g., "Source: whitepaper.pdf, Page 3").
            3. Keep the answer professional and concise.
            """
        ),
        HumanMessagePromptTemplate.from_template(
            """
            Context:
            {context}

            Question:
            {question}
            """
        )
    ]
)

model = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b"
)

parser = StrOutputParser()

def format_docs(docs):
  formatted_texts = []
  for i, doc in enumerate(docs):
    source = doc.metadata.get('source', 'Unknown Source')
    page = doc.metadata.get('page', 'N/A')

    entry = f"--- Document {i+1} (Source: {source}, Page: {page}) ---\nContent: {doc.page_content}\n"
    formatted_texts.append(entry)

  return "\n\n".join(formatted_texts)

rag_chain = (
    {"context": retriever|format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | model
    | parser
)

## 7. Execution
Run the RAG chain on a list of questions to see how it performs.

In [13]:
questions = [
    "What are the failure modes of Naïve RAG mentioned in the text?",

    "Summarize the main differences between a Standard LLM and an Agentic RAG system.",

    "How does the 'Self-Correction' mechanism work in Agentic RAG?",

    "Why is Prompt Engineering considered insufficient for enterprise applications according to the document?",

    "What is the best recipe for making Italian Pizza?"
]

for q in questions:
    print(f"--- Question: {q} ---")
    response = rag_chain.invoke(q)
    print(f"Answer: {response}")
    print("\n" + "="*50 + "\n")

--- Question: What are the failure modes of Naïve RAG mentioned in the text? ---
Answer: I don't know.  
Source: Document 1, Page 1.


--- Question: Summarize the main differences between a Standard LLM and an Agentic RAG system. ---
Answer: **Standard LLM**

* Operates only on **parametric memory** that is frozen at the end of training.  
* Cannot access real‑time or post‑cutoff data and therefore suffers from **knowledge cutoff** and frequent **hallucinations**.  
* Generates output in a single pass with no external retrieval step.  

**Agentic RAG**

* Extends the basic RAG pipeline (which adds a **non‑parametric vector database** for retrieval) with an **active, looping agent** that can **critique its own retrieval, rewrite queries, and select alternative data sources**.  
* Implements patterns such as the **“Grader”** for self‑correction, turning the system from a passive retrieval engine into an **active reasoning engine** with planning, memory, tool use, and self‑correction.  
*